# Convolutional Neural Networks in PyTorch

In this lab you will explore how to create a convolutional neural network (CNN) with PyTorch.

The CNN will run slowly unless you have an accelerator (like on newer MacBooks or in Google Colab).  On Colab, make sure to select the GPU runtime in `Runtime > Change runtime type`.

Here I load the Intel Image Classification dataset.  The images have been resized to 32x32.

In [ ]:
import os
if not os.path.exists('intel_image_classification.npz'):
  !wget -O intel_image_classification.npz "https://www.dropbox.com/scl/fi/g8piaiw6njhogb7fnu7b3/intel_image_classification.npz?rlkey=8ytc0ucpc7tg2gzs0gxtnzun5&dl=1"  

In [ ]:
import numpy as np
from matplotlib import pyplot as plt

In [ ]:
data = np.load('intel_image_classification.npz')
label_names = data['label_names']
X_train = data['X_train']
y_train = data['y_train']
X_test = data['X_test']
y_test = data['y_test']

The dataset has six categories and around 16k images total.

In [ ]:
label_names

In [ ]:
X_train.shape, X_train.dtype, X_train.min(), X_train.max()

In [ ]:
y_train.shape, y_train.dtype, y_train.min(), y_train.max()

In [ ]:
len(X_train), len(X_test)

In [ ]:
plt.imshow(X_train[0])
plt.title(y_train[0])

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

In [ ]:
X_train_torch = torch.tensor(X_train.transpose((0,3,1,2))) # change dim ordering for PyTorch
y_train_torch = torch.tensor(y_train)
X_test_torch = torch.tensor(X_test.transpose((0,3,1,2))) # change dim ordering for PyTorch
y_test_torch = torch.tensor(y_test)

## Exercises

Create a CNN to classify the images.  Here is a starting point for your architecture, although you are free to put whatever you want into the network:

* Input layer
* 2D convolution, 3x3 kernel, 32 channels, ReLU activation
* Max pooling: 2x2 kernel, stride of 2
* 2D convolution, 3x3 kernel, 64 channels, ReLU activation
* Max pooling: 2x2 kernel, stride of 2
* 2D convolution, 3x3 kernel, 128 channels, ReLU activation
* Max pooling: 2x2 kernel, stride of 2
* Flatten
* Linear layer: 6 outputs

Choose some settings for the optimizer etc. and see what accuracy can you achieve on this dataset.

In [ ]:
def compute_model_acc(model: torch.nn.Sequential, loader: DataLoader) -> float:
    num_correct = 0
    n = 0
    
    for X_batch, y_batch in loader:
        z_batch = model(X_batch)
        y_predict = torch.argmax(z, dim=1)
        num_correct += torch.sum(y_predict == y_batch)
        n += len(y_batch)
        
    return num_correct / n

In [ ]:
train_ds = TensorDataset(X_train_torch, y_train_torch)
test_ds = TensorDataset(X_test_torch, y_test_torch)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)

In [ ]:
model = nn.Sequential(
    nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2, stride=2),
    
    nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2, stride=2),

    nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2, stride=2),

    nn.Flatten(),

    nn.Linear(128 * 4 * 4, 6)
)

In [ ]:
loss_fn = torch.nn.CrossEntropyLoss()
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

epochs = 100
for epoch in range(epochs):
    model.train()
    for X_batch, y_batch in train_loader:
        opt.zero_grad() # zero out the gradients

        z_batch = model(X_batch) # compute z values
        loss = loss_fn(z_batch,y_batch) # compute loss

        loss.backward() # compute gradients

        opt.step() # apply gradients

    model.eval()
    print(f'epoch {epoch}: loss is {loss.item():.4f} -- training accuracy is {compute_model_acc(model, train_loader):.4f}, test accuracy is {compute_model_acc(model, test_loader):.4f}')